# Pipeline RAG complet

In [1]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters chromadb sentence-transformers transformers torch langchain-huggingface


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

c:\Users\Administrateur\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
raw_documents = [
    {
        "content": """
        # Introduction au Machine Learning
        
        Le Machine Learning (ML) est une branche de l'intelligence artificielle 
        qui permet aux systèmes d'apprendre et de s'améliorer automatiquement 
        à partir de l'expérience sans être explicitement programmés.
        
        ## Types de Machine Learning
        
        Il existe trois principaux types de ML:
        - Apprentissage supervisé: le modèle apprend à partir de données étiquetées
        - Apprentissage non supervisé: le modèle trouve des patterns dans des données non étiquetées
        - Apprentissage par renforcement: le modèle apprend par essai-erreur avec des récompenses
        
        ## Applications courantes
        
        Le ML est utilisé dans de nombreux domaines:
        - Reconnaissance d'images et de voix
        - Recommandation de produits
        - Détection de fraudes
        - Véhicules autonomes
        """,
        "metadata": {"source": "ml_intro.md", "category": "ml"}
    },
    {
        "content": """
        # RAG - Retrieval-Augmented Generation
        
        RAG est une technique qui améliore les réponses des LLM en combinant 
        la recherche documentaire avec la génération de texte.
        
        ## Comment fonctionne RAG ?
        
        1. L'utilisateur pose une question
        2. Le système recherche les documents pertinents dans une base de connaissances
        3. Les documents trouvés sont ajoutés au contexte du LLM
        4. Le LLM génère une réponse basée sur ce contexte enrichi
        
        ## Avantages de RAG
        
        - Réponses plus précises et à jour
        - Réduction des hallucinations
        - Possibilité de citer les sources
        - Pas besoin de réentraîner le modèle
        
        ## Composants clés
        
        - Embeddings: représentations vectorielles du texte
        - Vector Store: base de données pour recherche sémantique
        - Retriever: composant qui trouve les documents pertinents
        - LLM: modèle qui génère la réponse finale
        """,
        "metadata": {"source": "rag_guide.md", "category": "rag"}
    },
    {
        "content": """
        # Bases de Données Vectorielles
        
        Les bases de données vectorielles sont optimisées pour stocker 
        et rechercher des vecteurs de haute dimension (embeddings).
        
        ## Pourquoi utiliser une base vectorielle ?
        
        Les recherches traditionnelles par mots-clés ne capturent pas 
        le sens sémantique. Les bases vectorielles permettent de trouver 
        des contenus similaires même avec des formulations différentes.
        
        ## Bases populaires
        
        - ChromaDB: simple et embedded, idéal pour débuter
        - Pinecone: service cloud scalable
        - Weaviate: self-hosted avec recherche hybride
        - Qdrant: haute performance en Rust
        - FAISS: librairie Meta pour la recherche rapide
        
        ## Index et algorithmes
        
        Les index comme HNSW permettent des recherches approximatives 
        très rapides (ANN - Approximate Nearest Neighbors).
        """,
        "metadata": {"source": "vector_db.md", "category": "database"}
    }
]

# Convertir en Documents LangChain
documents = [
    Document(page_content=doc["content"], metadata=doc["metadata"])
    for doc in raw_documents
]

1. Découpage en chunks

In [4]:
# Créer le splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". "]
)

chunks = text_splitter.split_documents(documents)

print(f"Nombre de chunks : {len(chunks)}")
print(f"Contenu: {chunks[0].page_content[:100]}")

Nombre de chunks : 7
Contenu: # Introduction au Machine Learning

        Le Machine Learning (ML) est une branche de l'intelligen


2. Création du vector store

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_demo"
)

C:\Users\Administrateur\AppData\Local\Temp\ipykernel_6800\3694003834.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9241.11it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


3. Configuration du retriever

In [6]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

query = "Comment fonctionne RAG ?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(doc)
    print(f"[{i+1}] source : {doc.metadata.get('source')}")

page_content='# RAG - Retrieval-Augmented Generation

        RAG est une technique qui améliore les réponses des LLM en combinant 
        la recherche documentaire avec la génération de texte.

        ## Comment fonctionne RAG ?

        1. L'utilisateur pose une question
        2. Le système recherche les documents pertinents dans une base de connaissances
        3. Les documents trouvés sont ajoutés au contexte du LLM
        4. Le LLM génère une réponse basée sur ce contexte enrichi' metadata={'category': 'rag', 'source': 'rag_guide.md'}
[1] source : rag_guide.md
page_content='## Avantages de RAG

        - Réponses plus précises et à jour
        - Réduction des hallucinations
        - Possibilité de citer les sources
        - Pas besoin de réentraîner le modèle

        ## Composants clés

        - Embeddings: représentations vectorielles du texte
        - Vector Store: base de données pour recherche sémantique
        - Retriever: composant qui trouve les documents perti

4. Template de prompt

In [7]:
RAG_PROMPT_TEMPLATE = """
Tu es un assistant qui répond aux questions en utilisant uniquement 
le contexte fourni ci-dessous. Si l'information n'est pas dans le contexte, 
dis clairement que tu ne peux pas répondre avec les informations disponibles.

Contexte:
{context}

Question: {question}

Instructions:
- Réponds de manière concise et précise
- Cite les sources quand c'est pertinent
- Si tu n'es pas sûr, indique-le

Réponse:
"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

5. Pipeline sans llm externe

In [8]:
def simple_rag_pipeline(question, retriever, prompt_template):
    docs = retriever.invoke(question)

    context_parts = []
    sources = []

    for doc in docs:
        context_parts.append(doc.page_content)
        sources.append(doc.metadata.get('source', 'Unknown'))

    context = "\n\n --- \n\n".join(context_parts)

    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    return {
        "prompt": formatted_prompt,
        "context": context,
        "sources": list(sources),
    }

results = simple_rag_pipeline(
    "Comment fonctionne RAG ?",
    retriever,
    prompt
)

print("Prompt qui serait envoyé au LLM : \n")
print(results["prompt"])
print(f"source utilisées : {results['sources']}")

Prompt qui serait envoyé au LLM : 

Human: 
Tu es un assistant qui répond aux questions en utilisant uniquement 
le contexte fourni ci-dessous. Si l'information n'est pas dans le contexte, 
dis clairement que tu ne peux pas répondre avec les informations disponibles.

Contexte:
# RAG - Retrieval-Augmented Generation

        RAG est une technique qui améliore les réponses des LLM en combinant 
        la recherche documentaire avec la génération de texte.

        ## Comment fonctionne RAG ?

        1. L'utilisateur pose une question
        2. Le système recherche les documents pertinents dans une base de connaissances
        3. Les documents trouvés sont ajoutés au contexte du LLM
        4. Le LLM génère une réponse basée sur ce contexte enrichi

 --- 

## Avantages de RAG

        - Réponses plus précises et à jour
        - Réduction des hallucinations
        - Possibilité de citer les sources
        - Pas besoin de réentraîner le modèle

        ## Composants clés

        - 

Pipeline avec LLM

In [11]:
import torch
from langchain_huggingface import HuggingFacePipeline

def create_llm(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True
    )
   
    # Créer le pipeline
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.95,
        repetition_penalty=1.15
    )
   
    # Wrapper LangChain
    llm = HuggingFacePipeline(pipeline=pipe)

    return llm

llm = create_llm("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 202.48it/s]
Passing `generation_config` together with generation-related arguments=({'temperature', 'repetition_penalty', 'max_new_tokens', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
def full_rag_pipeline(question, retriever, llm, prompt_template):
    docs = retriever.invoke(question)

    context_parts = []
    sources = []

    for doc in docs:
        context_parts.append(doc.page_content)
        sources.append(doc.metadata.get('source', 'Unknown'))

    context = "\n\n --- \n\n".join(context_parts)

    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)


    return {
        "response": response,
        "sources" : list(set(sources)),
    }

results = full_rag_pipeline("Comment fonctionne RAG ?", retriever, llm, prompt)
print(results["response"])
print(results["sources"])
    

Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Human: 
Tu es un assistant qui répond aux questions en utilisant uniquement 
le contexte fourni ci-dessous. Si l'information n'est pas dans le contexte, 
dis clairement que tu ne peux pas répondre avec les informations disponibles.

Contexte:
# RAG - Retrieval-Augmented Generation

        RAG est une technique qui améliore les réponses des LLM en combinant 
        la recherche documentaire avec la génération de texte.

        ## Comment fonctionne RAG ?

        1. L'utilisateur pose une question
        2. Le système recherche les documents pertinents dans une base de connaissances
        3. Les documents trouvés sont ajoutés au contexte du LLM
        4. Le LLM génère une réponse basée sur ce contexte enrichi

 --- 

## Avantages de RAG

        - Réponses plus précises et à jour
        - Réduction des hallucinations
        - Possibilité de citer les sources
        - Pas besoin de réentraîner le modèle

        ## Composants clés

        - Embeddings: représentations vectorie